# Indoor Scene Classification (67 classes)

A 67-class indoor scene image classifier built on EfficientNetB0 transfer learning, with honest evaluation, Grad-CAM interpretability, and a live demo on Hugging Face Spaces.

**Dataset:** MIT Indoor Scenes (CVPR 2009), 67 categories, 15,620 images, downloaded from Kaggle (`itsahmad/indoor-scenes-cvpr-2019`).

**Headline result:** 79.92% top-1 and 96.51% top-5 accuracy on a random 80/20 split of the full dataset. This is not the official MIT Indoor-67 benchmark split, so the numbers should not be compared directly to published Indoor-67 results.

The notebook runs top to bottom in Colab: it downloads the data, trains in two stages, evaluates honestly, and produces the training curves, confusion matrix, sample predictions, and Grad-CAM figures.

## 1. Setup

Authenticate with Kaggle and set the output and checkpoint paths. Outputs are written to the Colab runtime, so running this notebook needs no Google Drive permission.

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('Kaggle')
!pip -q install --upgrade "kaggle>=1.8.0"

# Outputs go to the Colab runtime (no Drive mount, no permission prompt).
# They are ephemeral, so download them at the end before the session closes.
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(OUTPUT_DIR, "indoor_scene_cnn_best.keras")

## 2. Data

Download the dataset into the runtime (re-downloaded each session) and remove any corrupt image before the pipeline enumerates the directory.

In [ ]:
import glob
import tensorflow as tf

DATA_DIR = "/content/data/indoorCVPR_09/Images"

# Download into the runtime, re-downloaded each session (about 1 min)
!kaggle datasets download -d itsahmad/indoor-scenes-cvpr-2019 -p /content/data --unzip

# Remove malformed image(s) before any dataset enumerates the directory
bad = []
for p in glob.glob(os.path.join(DATA_DIR, "*", "*")):
    try:
        tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
    except Exception:
        bad.append(p)
        os.remove(p)

print("removed", len(bad), "corrupt files")
print(len(os.listdir(DATA_DIR)), "categories")   # must read 67

## 3. Dataset pipeline

Build the train and validation splits. Both use the same seed and `shuffle=True` so the split is complementary and all 67 classes appear in each. (A `shuffle=False` validation set silently leaks and inflates the score.)

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, labels="inferred", label_mode="int", color_mode="rgb",
    image_size=IMG_SIZE, batch_size=BATCH,
    validation_split=0.2, subset="training",
    seed=SEED, shuffle=True,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, labels="inferred", label_mode="int", color_mode="rgb",
    image_size=IMG_SIZE, batch_size=BATCH,
    validation_split=0.2, subset="validation",
    seed=SEED, shuffle=True,   # must match train: same seed and shuffle give a complementary split, all 67 classes
)

class_names = train_ds.class_names
print("classes:", len(class_names))   # must print 67

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 4. Model and Stage A: train the head on a frozen backbone

EfficientNetB0 with ImageNet weights, frozen, plus a small classification head trained at a higher learning rate. EfficientNet normalizes internally, so images are fed as raw 0-255 with no rescaling.

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

tf.keras.utils.set_random_seed(SEED)   # reproducible head init and dropout
NUM_CLASSES = len(class_names)         # 67

base = EfficientNetB0(include_top=False, weights="imagenet",
                      input_shape=(224, 224, 3))
base.trainable = False                 # frozen for Stage A

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs, training=False)       # inference mode; no rescale, EfficientNet normalizes internally
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy",
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")],
)

cb_a = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(CKPT_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
]
history_a = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cb_a)

## 5. Stage B: fine-tune the top blocks

Unfreeze from `block6a_expand_conv` onward at a low learning rate, keeping all BatchNorm layers frozen, and only overwrite the checkpoint if validation accuracy improves on Stage A.

In [ ]:
base.trainable = True
unfreeze = False
for layer in base.layers:
    if layer.name == "block6a_expand_conv":
        unfreeze = True
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False        # critical: never update BN during fine-tune
    else:
        layer.trainable = unfreeze

n_train = sum(1 for l in base.layers if l.trainable)
print(f"trainable base layers: {n_train} / {len(base.layers)}")   # expect 59 / 238

model.compile(                          # recompile required after changing trainable flags
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy",
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")],
)

best_a = max(history_a.history["val_accuracy"])   # only overwrite the checkpoint if we beat Stage A
cb_b = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(CKPT_PATH, monitor="val_accuracy", save_best_only=True,
                                       initial_value_threshold=best_a, verbose=1),
]
history_b = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cb_b)

## 6. Evaluation

Load the best checkpoint and report top-1, top-5, and a per-class classification report on the held-out validation set.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

best_model = tf.keras.models.load_model(CKPT_PATH)   # also verifies the saved file

y_true, y_prob = [], []
for x_batch, y_batch in val_ds:          # single loop keeps images and labels aligned
    y_prob.append(best_model.predict(x_batch, verbose=0))
    y_true.append(y_batch.numpy())
y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)
y_pred = y_prob.argmax(axis=1)

print("classes present in val:", len(np.unique(y_true)), "(should be 67)")
print(f"Validation images: {len(y_true)}")
print(f"Top-1: {(y_pred == y_true).mean():.4f}")
print(f"Top-5: {tf.keras.metrics.sparse_top_k_categorical_accuracy(y_true, y_prob, k=5).numpy().mean():.4f}")
print()
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

## 7. Training curves

Accuracy and loss across both stages, with the fine-tuning boundary marked.

In [ ]:
import matplotlib.pyplot as plt

# Stitch the two stages end to end
acc      = history_a.history["accuracy"]     + history_b.history["accuracy"]
val_acc  = history_a.history["val_accuracy"] + history_b.history["val_accuracy"]
loss     = history_a.history["loss"]         + history_b.history["loss"]
val_loss = history_a.history["val_loss"]     + history_b.history["val_loss"]
split    = len(history_a.history["accuracy"])   # epoch where fine-tuning began
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs, acc, label="train")
ax1.plot(epochs, val_acc, label="val")
ax1.axvline(split + 0.5, color="gray", ls="--", lw=1, label="fine-tune start")
ax1.set_title("Accuracy"); ax1.set_xlabel("epoch"); ax1.set_ylabel("accuracy")
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs, loss, label="train")
ax2.plot(epochs, val_loss, label="val")
ax2.axvline(split + 0.5, color="gray", ls="--", lw=1, label="fine-tune start")
ax2.set_title("Loss"); ax2.set_xlabel("epoch"); ax2.set_ylabel("loss")
ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle("Indoor Scene CNN: EfficientNetB0 two-stage transfer learning", y=1.02)
plt.tight_layout()
out = f"{OUTPUT_DIR}/training_curves.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print("saved:", out)
plt.show()

## 8. Confusion matrix

Row-normalized confusion matrix and the top confused class pairs, which show the errors are semantically adjacent rather than random.

In [ ]:
from sklearn.metrics import confusion_matrix

# Reuses y_true, y_pred, class_names from the evaluation cell
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)   # row-normalized gives per-class recall

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm_norm, cmap="viridis", vmin=0, vmax=1)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="fraction of true class")
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=90, fontsize=6)
ax.set_yticklabels(class_names, fontsize=6)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Confusion matrix (row-normalized), 67-class held-out val")
plt.tight_layout()
out = f"{OUTPUT_DIR}/confusion_matrix.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print("saved:", out)
plt.show()

# Top off-diagonal confusions are the honest-eval narrative
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
pairs = [(cm_off[i, j], class_names[i], class_names[j])
         for i in range(len(class_names)) for j in range(len(class_names))
         if cm_off[i, j] > 0]
pairs.sort(reverse=True)
print("\nTop 12 confusions (true -> predicted, count):")
for count, t, p in pairs[:12]:
    print(f"  {count:3d}   {t}  ->  {p}")

## 9. Sample predictions

Predictions with confidence on a batch of held-out images. Green titles are correct, red are misses.

In [ ]:
# One batch from the val set (images are float 0-255, not rescaled)
images, labels = next(iter(val_ds))
probs = best_model.predict(images, verbose=0)
preds = probs.argmax(axis=1)
conf  = probs.max(axis=1)

fig, axes = plt.subplots(3, 5, figsize=(16, 10))   # 3 rows x 5 cols
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].numpy().astype("uint8"))
    ax.axis("off")
    true_name = class_names[labels[i].numpy()]
    pred_name = class_names[preds[i]]
    correct   = (preds[i] == labels[i].numpy())
    color     = "green" if correct else "red"
    ax.set_title(f"pred: {pred_name} ({conf[i]:.0%})\ntrue: {true_name}",
                 color=color, fontsize=9)
plt.suptitle("Sample predictions (green = correct, red = miss)", y=1.0, fontsize=13)
plt.tight_layout()
out = f"{OUTPUT_DIR}/sample_predictions.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print("saved:", out)
plt.show()

## 10. Grad-CAM interpretability

Class-activation overlays showing which regions drive each prediction. The conv features live inside the nested `efficientnetb0` sub-model, so the grad model is rooted at the sub-model input and the head layers are re-threaded onto its output to keep one connected graph.

In [ ]:
import matplotlib.cm as cm

# Conv features live in the nested sub-model. Root the grad model at the
# sub-model's own input, then re-thread the head layers onto its output so
# all tensors share one connected graph.
base_submodel = best_model.get_layer("efficientnetb0")
conv_output = base_submodel.output          # (None, 7, 7, 1280), inner graph
x = conv_output
for layer in best_model.layers[2:]:         # GAP -> Dropout -> Dense, in order
    x = layer(x)
grad_model = tf.keras.models.Model(base_submodel.input, [conv_output, x])

def gradcam_heatmap(img_batch, pred_index=None):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_score = preds[:, pred_index]
    grads = tape.gradient(class_score, conv_out)          # d(score) / d(feature map)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))        # importance per channel
    conv_out = conv_out[0]
    heat = conv_out @ pooled[..., tf.newaxis]             # weighted sum of channels
    heat = tf.squeeze(heat)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)   # relu then normalize
    return heat.numpy()

# Pull one batch (raw 0-255, not rescaled, same as training)
images, labels = next(iter(val_ds))
probs = best_model.predict(images, verbose=0)
preds = probs.argmax(axis=1)

N = 4
fig, axes = plt.subplots(N, 2, figsize=(8, 4 * N))
for i in range(N):
    img = images[i].numpy().astype("uint8")
    heat = gradcam_heatmap(images[i:i+1])
    # Resize heatmap to image size and colorize
    heat_resized = tf.image.resize(heat[..., np.newaxis], (224, 224)).numpy().squeeze()
    heat_color = cm.jet(heat_resized)[..., :3]
    overlay = (0.6 * img / 255.0 + 0.4 * heat_color)
    true_name = class_names[labels[i].numpy()]
    pred_name = class_names[preds[i]]
    color = "green" if preds[i] == labels[i].numpy() else "red"
    axes[i, 0].imshow(img); axes[i, 0].axis("off")
    axes[i, 0].set_title(f"true: {true_name}", fontsize=10)
    axes[i, 1].imshow(overlay); axes[i, 1].axis("off")
    axes[i, 1].set_title(f"pred: {pred_name} ({probs[i].max():.0%})",
                         color=color, fontsize=10)
plt.suptitle("Grad-CAM: where the model looks (red = high attention)", y=1.0, fontsize=13)
plt.tight_layout()
out = f"{OUTPUT_DIR}/gradcam.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print("saved:", out)
plt.show()

## Save the generated files

The model and figures live in the Colab runtime, which is wiped when the session ends. Run this to list them and download a zip you can commit to the repo.

In [ ]:
import shutil
from google.colab import files

print("files in", OUTPUT_DIR + ":")
for name in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, name)) // 1024
    print(f"  {name} ({size_kb} KB)")

archive = shutil.make_archive("/content/indoor_scene_cnn_outputs", "zip", OUTPUT_DIR)
files.download(archive)